# 02. 特徴量重要度分析

LightGBMで勝利予測モデルを学習し、SHAP値で各ファクターの寄与度を可視化する。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
import shap
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import roc_auc_score, log_loss
from pathlib import Path
import sys; sys.path.append('../src')
from evaluate_roi import roi_from_model_proba, kelly_bet_sizes

plt.rcParams['figure.figsize'] = (12, 6)
sns.set_theme(style='whitegrid')

df = pd.read_csv('../data/processed/features.csv', parse_dates=['race_date'])
df = df.sort_values('race_date').reset_index(drop=True)
print(df.shape)

In [ ]:
FEATURES = [
    'log_odds', 'odds_rank_pct', 'prob_norm',
    'popularity',
    'age', 'sex_code',
    'weight', 'weight_diff', 'weight_abs_diff', 'weight_increase', 'weight_decrease',
    'log_distance', 'is_sprint', 'is_mile', 'is_middle', 'is_long',
    'surface_code', 'condition_code',
    'jockey_te_place', 'trainer_te_place', 'course_te_place',
    'course_id', 'jockey_id', 'trainer_id',
]
TARGET = 'label_win'

X = df[FEATURES]
y = df[TARGET]
print('正例率:', y.mean())

## 時系列クロスバリデーション

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)
oof_proba = np.zeros(len(df))
models = []
auc_scores = []

params = {
    'objective': 'binary',
    'metric': 'auc',
    'learning_rate': 0.05,
    'num_leaves': 63,
    'min_child_samples': 50,
    'colsample_bytree': 0.8,
    'subsample': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'verbose': -1,
    'n_estimators': 500,
    'early_stopping_rounds': 50,
}

for fold, (train_idx, val_idx) in enumerate(tscv.split(X)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = lgb.LGBMClassifier(**params)
    model.fit(X_tr, y_tr,
              eval_set=[(X_val, y_val)],
              callbacks=[lgb.log_evaluation(period=100)])

    proba = model.predict_proba(X_val)[:, 1]
    oof_proba[val_idx] = proba
    auc = roc_auc_score(y_val, proba)
    auc_scores.append(auc)
    models.append(model)
    print(f'Fold {fold+1} AUC: {auc:.4f}')

print(f'\nOOF AUC: {np.mean(auc_scores):.4f} ± {np.std(auc_scores):.4f}')

## LightGBM Feature Importance

In [ ]:
fi_gain = pd.Series(np.mean([m.booster_.feature_importance('gain') for m in models], axis=0),
                     index=FEATURES).sort_values(ascending=False)
fi_split = pd.Series(np.mean([m.booster_.feature_importance('split') for m in models], axis=0),
                      index=FEATURES).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fi_gain.head(15).plot(kind='barh', ax=axes[0], title='Feature Importance (Gain)')
fi_split.head(15).plot(kind='barh', ax=axes[1], title='Feature Importance (Split)', color='orange')
for ax in axes:
    ax.invert_yaxis()
plt.tight_layout()
plt.show()

## SHAP値による寄与度分析

In [ ]:
# 最後のfoldのモデルでSHAP計算
last_model = models[-1]
last_val_idx = list(tscv.split(X))[-1][1]
X_sample = X.iloc[last_val_idx]

explainer = shap.TreeExplainer(last_model)
shap_values = explainer.shap_values(X_sample)
if isinstance(shap_values, list):
    shap_values = shap_values[1]  # 勝利クラス

plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_sample, feature_names=FEATURES, show=False)
plt.title('SHAP Summary Plot（各特徴量の予測への寄与）')
plt.tight_layout()
plt.show()

In [ ]:
# SHAP Bar Plot（絶対平均）
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_sample, feature_names=FEATURES, plot_type='bar', show=False)
plt.title('SHAP Bar Plot（平均絶対SHAP値）')
plt.tight_layout()
plt.show()

## SHAP 依存プロット（重要特徴量）

In [ ]:
top_features = fi_gain.head(4).index.tolist()
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, feat in zip(axes.flat, top_features):
    feat_idx = list(FEATURES).index(feat)
    ax.scatter(X_sample[feat], shap_values[:, feat_idx], alpha=0.3, s=8)
    ax.set_xlabel(feat)
    ax.set_ylabel('SHAP value')
    ax.set_title(f'{feat} の SHAP 依存プロット')
    ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
plt.tight_layout()
plt.show()

## OOFスコアを保存

In [ ]:
df['pred_win_proba'] = oof_proba
df.to_csv('../data/processed/features_with_pred.csv', index=False, encoding='utf-8-sig')
print('保存完了: data/processed/features_with_pred.csv')